# Homework 1 - data validation & cleaning

In short, the main task is to clean The Metropolitan Museum of Art Open Access dataset.
  
> The instructions are not given in detail: It is up to you to come up with ideas on how to fulfill the particular tasks as best as possible!

However, we **strongly recommend and require** the following:
* Follow the assignment step by step. Number each step.
* Most steps contain the number of features that should be treated. You can preprocess more features. However, it does not mean the teacher will give you more points. Focus on quality, not quantity.
* Properly comment on all your steps. Use Markdown cells and visualizations. Comments are evaluated for 2 points of the total, together with the final presentation of the solution. However, it is not desirable to write novels! 
* This task is timewise and computationally intensive. Do not leave it to the last minute.
* Hand in a notebook that has already been run (i.e., do not delete outputs before handing in).

## What are you supposed to do:

  1. Download the dataset MetObjects.csv from the repository https://github.com/metmuseum/openaccess/.
  1. Check consistency (i.e., that the same things are represented in the same way) of at least **three features** where you expect problems (including the "Object Name" feature). You can propose how to clean the selected features. However, **do not apply cleaning** (in your interest) 🙂 _(1.5 points)_
  1. Select at least **two features** (i.e., one couple) where you expect integrity problems (describe your choice) and check the integrity of those features. By integrity, we mean correct logical relations between features (e.g., female names for females only). _(2 points)_
  1. Convert at least **five features** to a proper data type. Choose at least one numeric, one categorical (i.e., ordinal or nominal), and one datetime. _(1.5 points)_
  1. Find some outliers and describe your method. _(3 points, depends on creativity)_
  1. Detect missing data in at least **three features**, convert them to a proper representation (if they are already not), and impute missing values in at least **one feature** using some imputation method (i.e., imputation by mean or median is too trivial to obtain any points). _(2 + 3 points, depends on creativity)_
  1. Focus more precisely on cleaning the "Medium" feature. As if you were to use it in the KNN classification algorithm later. _(3 points)_
  1. Focus on the extraction of the physical dimensions of each item (width, depth, and height in centimeters) from the "Dimensions" feature. _(2 points)_
  
All your steps, your choices of methods, and the following code **must be commented on!** For text comments (discussion, etc., not code comments), use **Markdown cells**. Comments are evaluated for 2 points together with the final presentation of the solution. 

**If you do all this properly, you will obtain 20 points.**

## Comments

  * Please follow the technical instructions from https://courses.fit.cvut.cz/NI-PDD/homeworks/index.html.
  * Methods that are more complex and were not shown during the tutorials are considered more creative and should be described in detail.
  * English is not compulsory.

In [1]:
import numpy as np
import pandas as pd
import sklearn as skit
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from scipy.stats import chi2_contingency

import seaborn as sns
import re

import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
df = pd.read_csv('openaccess/MetObjects.csv', sep=',', dtype=object)

## Consistency

Data should be represented the same way for all objects.

**Medium**:

This feature should describe what kind of material the art piece has been created on or from (gold, glass, linen, ...). Often, times the feature instead describes the art piece (drawing, painting, fan, ...), is unnecessarily descriptive (portfolio of 10 lithographs by rauschenberg ...) or misspelled (wtaercolor). 

In [3]:
df_tmp = df[['Object Name', 'Title', 'Medium', 'Classification']].iloc[[1,320,262647,30298,33723,136637,484891]].copy()
df_tmp

,Object Name,Title,Medium,Classification
1,Coin,Ten-dollar Liberty Head Coin,Gold,NaN
320,Door,"Batten Door from the Williams House, near Pres...",Pine,NaN
262647,Painting,"Kenneth Callahan painting on cardboard, 1947",Painting,Paintings
30298,Hanging scroll,NaN,Hanging scroll; ink and color on silk,Paintings
33723,Incense burner,NaN,Porcelain painted with cobalt blue under trans...,Ceramics
136637,Apron fragment,Fragment from an apron (opreg cu ciucuri),"Background: twill weave (warp: Turcana wool, Z...",Textiles-Woven
484891,Drawing,"Family, the perfect Happiness (En Famille, le ...",Wtaercolor,Drawings


I would try to make the 'Medium' a feature that focuses on the main material used. Knowing the material should be important information as it is required for proper handling and storing of the art piece. Some of the information in 'Medium' is unnecessary and is repeated in 'Object Name' or 'Classification'. 'Object Name' sometimes has the material used written, while 'Medium' does not. But I would still make a feature 'Description' as some information may be lost in some cases.

In [4]:
df_tmp['Medium']      = df_tmp['Medium'].str.lower()
df_tmp['Medium New']  = ('gold', 'pine', 'cardboard', 'silk', 'porcelain', 'textile', 'watercolor') 

df_tmp = df_tmp[['Object Name', 'Title', 'Medium New', 'Medium', 'Classification']]
df_tmp.rename(columns = {'Medium' : 'Description (Old Medium)'}, inplace=True)

df_tmp

,Object Name,Title,Medium New,Description (Old Medium),Classification
1,Coin,Ten-dollar Liberty Head Coin,gold,gold,NaN
320,Door,"Batten Door from the Williams House, near Pres...",pine,pine,NaN
262647,Painting,"Kenneth Callahan painting on cardboard, 1947",cardboard,painting,Paintings
30298,Hanging scroll,NaN,silk,hanging scroll; ink and color on silk,Paintings
33723,Incense burner,NaN,porcelain,porcelain painted with cobalt blue under trans...,Ceramics
136637,Apron fragment,Fragment from an apron (opreg cu ciucuri),textile,"background: twill weave (warp: turcana wool, z...",Textiles-Woven
484891,Drawing,"Family, the perfect Happiness (En Famille, le ...",watercolor,wtaercolor,Drawings


**Gallery Number**:

Represents where the artpiece is displayed in a numeric 3 digit format. If missing, it's most likely being stored. But some of the values are written places.

In [5]:
df[['Object Name', 'Title', 'Gallery Number']].iloc[[1,122,10164]]

,Object Name,Title,Gallery Number
1,Coin,Ten-dollar Liberty Head Coin,NaN
122,Architectural elements,Architectural Elements,737
10164,Sculpture,Eagle,in Great Hall


In [6]:
df_tmp = df[~df['Gallery Number'].str.contains(r'\d', na=True)]
print(df_tmp['Gallery Number'].unique())

['in Great Hall' 'Petrie Ct. Café' 'on Fifth Avenue' 'Watson Library']


The Great Hall in the Metropolitan Museum of Art is not really a gallery, but the main entry point into the museum. For that reason, it doesn't have a 'Gallery Number'. But I could search for empty numbers, pick one and assign it to the Great Hall. Obviously, I can't just add any number as the numbers have a meaning behind them. Because it is the entry point, I will try to search for the lowest possible number.

![image.jpg](images/map.jpg)

In [7]:
count = df[df['Gallery Number'].str.contains('000', na=False)].shape[0]
print('Number of occurrences of Gallery Number \"000\": ', count)

df_tmp = df[['Object Number', 'Object Name', 'Title', 'Gallery Number']].iloc[[1,122,10164]]
df_tmp.iloc[1, 3] = '000'
df_tmp

Number of occurrences of Gallery Number "000":  0


,Object Number,Object Name,Title,Gallery Number
1,1980.264.5,Coin,Ten-dollar Liberty Head Coin,NaN
122,68.133.7,Architectural elements,Architectural Elements,000
10164,19.94.1,Sculpture,Eagle,in Great Hall


000 seems empty, so I could assign it to the Great Hall. I would repeat the same process with the other non-numbered places.

**Object Name**:

Object Name should describe the physical type of the art piece (photograph, bottle, picture, ...). There are many cases in which similar objects are described differently (words have different order), is too descriptive and some don't describe the object well (Four sons of Horus, not an object description).

Belt plate and loop or Belt loop and plate

![image.jpg](images/belts.jpg)

In [8]:
df_tmp = df[['Object Name', 'Title', 'Medium']].iloc[[314670, 314674, 340031, 340032, 365536, 365656]].copy()
df_tmp

,Object Name,Title,Medium
314670,Belt plate and loop,Plate and Loop of a Belt Buckle,"Iron, silver inlay, brass inlay"
314674,Belt loop and plate,Loop and Plate of a Belt Buckle,"Iron, silver and brass inlay"
340031,Watercolor of musician playing qin,Musician playing Guqin (古琴 ),Watercolor on paper
340032,Watercolor of musician playing mu yu,Watercolor of musician playing mu yu,Watercolor on paper
365536,"Bracelet with 4 wedjat eyes, 2 barrel beads, a...","Bracelet with 4 wedjat eyes, 2 barrel beads, a...","Linen, faience, gold, amethyst"
365656,"Four sons of Horus, Hapy, baboon-headed, Nesen...",Viscera figure with baboon head (Hapy),"Mud, wax"


Similar to 'Medium' I would try to make 'Object Name' try and describe the art piece as simply as possible, to help with identification. 'Title' can often times help with longer or more detailed descriptions of the object.

In [9]:
df_tmp['Object Name New'] = ('belt plate and loop', 'belt plate and loop', 'watercolor', 'watercolor', 'bracelet', 'figure')

df_tmp['Object Name']  = df_tmp['Object Name'].str.lower()
df_tmp.rename(columns = {'Object Name' : 'Object Name Old'}, inplace=True)

df_tmp = df_tmp[['Object Name New', 'Object Name Old', 'Title', 'Medium']]

df_tmp

,Object Name New,Object Name Old,Title,Medium
314670,belt plate and loop,belt plate and loop,Plate and Loop of a Belt Buckle,"Iron, silver inlay, brass inlay"
314674,belt plate and loop,belt loop and plate,Loop and Plate of a Belt Buckle,"Iron, silver and brass inlay"
340031,watercolor,watercolor of musician playing qin,Musician playing Guqin (古琴 ),Watercolor on paper
340032,watercolor,watercolor of musician playing mu yu,Watercolor of musician playing mu yu,Watercolor on paper
365536,bracelet,"bracelet with 4 wedjat eyes, 2 barrel beads, a...","Bracelet with 4 wedjat eyes, 2 barrel beads, a...","Linen, faience, gold, amethyst"
365656,figure,"four sons of horus, hapy, baboon-headed, nesen...",Viscera figure with baboon head (Hapy),"Mud, wax"


## Integrity

Correct logical relation between features.


I am going to look at 'Artist Begin Date', 'Artist End Date', 'Object Begin Date' and 'Object End Date'. The correct relation should look like this:

$$Artist Begin Date \leq Object Begin Date \leq Object End Date \leq Artist End Date$$

The artists features can be quite inconsistent, as it can represent a single artist, a group or even an entire workshop. Artist Begin and End Dates should represent the time period in which the artist lived or was active. While the Object Begin and End Date indicate in which years the work on the artwork started and ended.

Therefore, if the years don't match, it should be considered as an error. Of course, there still might be exceptions, art pieces might be finished by a different artist, restorations, copies and so on. 

In [10]:
df_tmp = df[['Object Number', 'Title', 'Artist Begin Date', 'Object Begin Date', 'Object End Date', 'Artist End Date']].copy()

df_tmp = df_tmp.loc[[35,494,112561,249903,1,10]].copy()

df_tmp['Integrity Violation'] = ('Artist Begin Date > Obejct Begin Date',
                                 'Artist Begin Date > Obejct End Date',
                                 'Artist Begin Date > Artist End Date',
                                 'Obejct Begin Date > Obejct End Date',
                                 'Obejct Begin Date > Artist End Date',
                                 'Obejct En dDate   > Artist End Date')

df_tmp

,Object Number,Title,Artist Begin Date,Object Begin Date,Object End Date,Artist End Date,Integrity Violation
35,1976.319,Side Chair,1885,1884,1887,1932,Artist Begin Date > Obejct Begin Date
494,1984.395.12,Book flask,1852,1849,1849,1852,Artist Begin Date > Obejct End Date
112561,58.62,Jeu de boules or The Ball Game (from the Story...,1582,1585,1605,1562,Artist Begin Date > Artist End Date
249903,2008.525,Study for a Tomb,1757,1785,1780,1822,Obejct Begin Date > Obejct End Date
1,1980.264.5,Ten-dollar Liberty Head Coin,1785,1901,1901,1844,Obejct Begin Date > Artist End Date
10,1979.486.2,Two-and-a-half-dollar Liberty Head Coin,1785,1907,1907,1844,Obejct En dDate > Artist End Date


Here are some examples of integrity violations. The book flask was stamped in 1849 by the Bennington Factory, but the maker of the flask is considered United States Pottery Company, which started in 1852. The stamp could bear the founding year of the Factory but was made by the Pottery Company for the Bennington Factory.

The second example is of a chair which has the signature of the artist, place and the year of 1887. But the artist is considered to be active from 1888. This chair could indicate that the artist was active before his known productive years.

I think both of these examined examples could be considered errors.

![IntegrityViolations.png](images/IntegrityViolations.png)

## Conversion

### Numeric

**Object ID**:

Should be a unique identifier for all items in the database. There aren't any missing, duplicate or non-numeric values, so I can simply convert it into `int`.

In [11]:
obj_nul = df['Object ID'].isnull().sum()
obj_dup = df['Object ID'].duplicated().sum()
df_tmp  = df[~df['Object ID'].str.contains(r'\d', na=True)]
obj_con = df_tmp['Object ID'].sum()
print( 'Object ID has', obj_nul, 'missing values,\n\t     ', obj_dup, 'duplicates,\n\t     ', obj_con, 'non-numeric values.\n' )

display(df[['Object ID']].sample(3))

df['Object ID'] = df['Object ID'].astype('int64')

Object ID has 0 missing values,
	      0 duplicates,
	      0 non-numeric values.



,Object ID
433688,741814
293995,433749
86438,125146


**Object Begin/End Date**:

Represents the approximate date the artwork could have been made. Both features seem to be consistent, have no missing values, but they do have duplicates and negative ones, which are expected. I am going to transform them to `int` as they contain only years despite the feature saying date. Trying to estimate the year in which the art piece was made is probably hard enough even without trying to get the month and day right.

In [12]:
obj_nul = df['Object Begin Date'].isnull().sum()
obj_dup = df['Object Begin Date'].duplicated().sum()
print( 'Object Begin Date has', obj_nul, 'missing values,\n\t\t     ', obj_dup, 'duplicates,' )

obj_nul = df['Object End Date'].isnull().sum()
obj_dup = df['Object End Date'].duplicated().sum()
print( '\nObject End Date has', obj_nul, 'missing values,\n\t\t   ', obj_dup, 'duplicates,\n' )

display(df[['Object Begin Date', 'Object End Date']].sample(3))

df['Object Begin Date'] = df['Object Begin Date'].astype('int64')
df['Object End Date'] = df['Object End Date'].astype('int64')

Object Begin Date has 0 missing values,
		      482880 duplicates,

Object End Date has 0 missing values,
		    482915 duplicates,



,Object Begin Date,Object End Date
79710,1860,1869
237216,1790,1800
164905,1900,1919


### Categorical

#### Nominal

Individual values don't have quantitative information or natural order.

**Department**:

Refers to the specific curatorial administration within the museum. It has no missing values, duplicates are ok and the data is consistent. 

In [13]:
obj_nul = df['Department'].isnull().sum()
obj_dup = df['Department'].duplicated().sum()
obj_cat = df['Department'].unique()
obj_nuq = df['Department'].nunique()
print( 'Department has', obj_nul, 'missing values,\n\t      ' , obj_dup, 'duplicates,\n\t      ', obj_nuq, 'categories.\n\n',   'The departments:')
for i in obj_cat: print( '\t', i )

department_category = pd.api.types.CategoricalDtype(categories=df['Department'].unique(), ordered=False)
df['Department'] = df['Department'].astype(department_category)

Department has 0 missing values,
	       484937 duplicates,
	       19 categories.

 The departments:
	 The American Wing
	 European Sculpture and Decorative Arts
	 Modern and Contemporary Art
	 Arms and Armor
	 Medieval Art
	 Asian Art
	 Islamic Art
	 Costume Institute
	 Arts of Africa, Oceania, and the Americas
	 Drawings and Prints
	 Greek and Roman Art
	 Photographs
	 Ancient Near Eastern Art
	 Egyptian Art
	 European Paintings
	 Robert Lehman Collection
	 The Cloisters
	 Musical Instruments
	 The Libraries


#### Dichotomous

Nominal variables which have only two categories.

**Is Public Domain**:

Tells us if the artwork is free for public work or is protected by copyright. There are no missing values, duplicates are ok and has only two values.

In [14]:
obj_nul = df['Is Public Domain'].isnull().sum()
obj_dup = df['Is Public Domain'].duplicated().sum()
obj_cat = df['Is Public Domain'].unique()
print( 'Is Public Domain has', obj_nul, 'missing values,\n\t\t    ', obj_dup, 'duplicates,\nCategories: ' )
for i in obj_cat: print( '\t', i )

bool_category = pd.api.types.CategoricalDtype(categories=['True', 'False'], ordered=False)
df['Is Highlight'] = df['Is Highlight'].astype(bool_category)

Is Public Domain has 0 missing values,
		     484954 duplicates,
Categories: 
	 False
	 True


**Is Highlight**:

Indicates if the artpiece is popular or an important in the collection. Values are same as in 'Is Public Domain.'

In [15]:
obj_nul = df['Is Highlight'].isnull().sum()
obj_dup = df['Is Highlight'].duplicated().sum()
obj_cat = df['Is Highlight'].unique()
print( 'Is Highlight has', obj_nul, 'missing values,\n\t        ', obj_dup, 'duplicates.')

bool_category = pd.api.types.CategoricalDtype(categories=['True', 'False'], ordered=False)
df['Is Highlight'] = df['Is Highlight'].astype(bool_category)

Is Highlight has 0 missing values,
	         484954 duplicates.


**Is Timeline Work**:

Whether the object is on the Timeline of Art History website or not. No missing values but duplicates and either `True` or `False`. Just like 'Is Highlight' or 'Is Public Domain'.

In [16]:
obj_nul = df['Is Public Domain'].isnull().sum()
obj_dup = df['Is Public Domain'].duplicated().sum()
obj_cat = df['Is Public Domain'].unique()
print( 'Is Public Domain has', obj_nul, 'missing values,\n\t\t    ', obj_dup, 'duplicates,\nCategories: ' )
for i in obj_cat: print( '\t', i )

bool_category = pd.api.types.CategoricalDtype(categories=['True', 'False'], ordered=False)
df['Is Public Domain'] = df['Is Public Domain'].astype(bool_category)

Is Public Domain has 0 missing values,
		     484954 duplicates,
Categories: 
	 False
	 True


#### Datetime

I am going to convert AccessionYear, even if the feature specifies years there are some that have dates. The other candidates for datetime would require a lot of cleaning up (Artist Date - lists all the artists, Object Date - string format and some values are negative, MetaData Date - has no values). Funily, the one feature that could be kept as a date, the museum should known the exact date when it acquired it's art pieces, is kept only in years. While those that need to be estimated are called dates, instead of years.

**AccessionYear**:

Represent the year the piece was added to the museum's collection. Some values are missing, duplicate are ok in this case. There are some values that also contain dates (format YYYY-MM-DD) so it's not consistent.

In [17]:
obj_nul = df['AccessionYear'].isnull().sum()
obj_dup = df['AccessionYear'].duplicated().sum()
tmp_df  = df[df['AccessionYear'].str.contains('-', na=False)]
obj_con = tmp_df['AccessionYear'].count()

print( 'AccessionYear has', obj_nul, 'missing values,\n\t\t ', obj_dup, 'duplicates,\n\t\t ', obj_con, 'non-numeric values.\n' )

print( 'Some years with dates.' )
display(tmp_df['AccessionYear'].sample(5))
#df['AccessionYear'] = df['AccessionYear'].str[:4]

# df['AccessionYear'] = df['AccessionYear'].astype('Int64')
df['AccessionYear'] = pd.to_datetime(df['AccessionYear'], format='mixed', yearfirst=True)

AccessionYear has 3862 missing values,
		  484775 duplicates,
		  40 non-numeric values.

Some years with dates.


481629    1960-11-15
481627    1959-10-16
338849    2022-05-20
481805    1962-10-15
481804    1962-10-15
Name: AccessionYear, dtype: object

#### Converted features

In [18]:
df[['Object ID', 'Object Begin Date', 'Object End Date', 'AccessionYear', 'Department', 'Is Public Domain', 'Is Highlight', 'Is Timeline Work' ]].dtypes

Object ID                     int64
Object Begin Date             int64
Object End Date               int64
AccessionYear        datetime64[ns]
Department                 category
Is Public Domain           category
Is Highlight               category
Is Timeline Work             object
dtype: object

## Outliers

Outliers are values that differ heavily from other values.

I would like to find out if there are any cultures that influenced other countries or artist to make pieces of art in their culture. Trying to do this across the whole dataframe would be quite difficult as there are many different values, varying pairing options (French - France, Japan - Japanese, American - United States, ...), missing values and data inconsistencies. I will start out with the most frequent culture.

In [19]:
cols = ["Object ID","Object Number","Object Name","Title","Culture","Country","Artist Nationality"]

df_tmp = df[cols].copy()

df_tmp['Culture']            = df_tmp['Culture'].str.lower()
df_tmp['Country']            = df_tmp['Country'].str.lower()
df_tmp['Artist Nationality'] = df_tmp['Artist Nationality'].str.lower()

In [20]:
df_tmp['Culture'] = df_tmp['Culture'].str.replace(r',.*|;.*|\(.*|\?', '', regex=True).str.strip()
df_tmp['Culture'] = df_tmp['Culture'].str.replace(r'\(.*', '', regex=True).str.strip()
df_tmp['Culture'] = df_tmp['Culture'].str.replace( 'possibly|probably|northern|southern|western|eastern|north|south|west|east', '', regex=True).str.strip()

culture_map = {
    'america'                : 'american',
    'american u.s.a.'        : 'american',
    'new york: [s.n.]'       : 'american',
    'colonial american'      : 'american',
    'boston: j. r. osgood'   : 'american',
    'american philadelphia'  : 'american',
    'new york: harper bros.' : 'american',
}

df_tmp['Culture'] = df_tmp['Culture'].replace(culture_map)

print(df_tmp['Culture'].value_counts().head(5))

Culture
american    29463
french      24008
greek       20845
japan       16985
china       13905
Name: count, dtype: int64


As The Metropolitan Museum is located in the United States, it's no surprise that the biggest portion of its collection is American. Next, I will take American culture and look at what is the country of origin of artworks.


The data is also very inconsistent. I am only interested in artworks that were made in other countries. So countries that have United States tagged, will be grouped with them.

In [21]:
df_cul = df_tmp[df_tmp['Culture'] == 'american'].copy()

country_map = {
    'usa'     : 'united states',
    'u.s.a.'  : 'united states',
    'america' : 'united states',
}

df_cul['Country'] = df_cul['Country'].replace(country_map)

df_cul_cou = df_cul[~df_cul['Country'].str.contains('united states', case=True, na=True, regex=False)].copy()

display(df_cul_cou[['Object ID', 'Object Number', 'Object Name', 'Title', 'Culture', 'Country']].loc[[1643, 4757, 463230, 17027]])
print('Number of artpieces with American culture made outside of United States: ', df_cul_cou['Country'].count())

,Object ID,Object Number,Object Name,Title,Culture,Country
1643,1793,68.20.1,Side Chair,Side Chair,american,japan
4757,5122,83.2.13,Medal,Eccleston Medal,american,united kingdom
463230,814649,"2018.648.1a, b",Vase,Vase (Old World),american,germany
17027,20558,2012.484.1a–v,Stool,Stool,american,india


Number of artpieces with American culture made outside of United States:  93


![image.jpg](images/outliers.jpg)


The chair was designed in America by an American, but it was manufactured in Jan.



The medal was designed and made in England, but it depicts the American president, George Wasngton.



The vases were made in Germany, but they were decorated by an American diplomat while he was stationedn Germany.



The last piece of art was made by an American living in India who created a unique combinion of styles.



But most data shows what Americans created in foreign countries. Now we will focus on the 'Artist Nationality' feature.

In [22]:
df_cul['Artist Nationality'] = df_cul['Artist Nationality'].replace(' ', '').str.strip()
df_cul['Artist Nationality'] = df_cul['Artist Nationality'].replace('', np.nan)
df_cul['Artist Nationality'] = df_cul['Artist Nationality'].replace('^\\|$', np.nan, regex=True)

df_cul_art = df_cul[~df_cul['Artist Nationality'].str.contains('american|united states', na=True, regex=True)].copy()

display(df_cul_art[["Object ID","Object Number","Object Name","Title","Culture","Artist Nationality"]].loc[[80218, 331603, 10447, 473150, 75633]])
print('Number of artpieces with American culture made by non-american artist: ', 
      df_cul_art['Artist Nationality'].count())

,Object ID,Object Number,Object Name,Title,Culture,Artist Nationality
80218,109788,1980.176,Fancy dress costume,Fancy dress costume,american,russian|french
331603,490992,45.134.2a,Sample,Piece,american,|austrian
10447,11363,16.27,Sculpture,Genius of Immortality,american,hungarian
473150,838139,2020.1,Window,Stained glass window,american,"british, scottish"
75633,98573,2004.20,Necklace,Necklace,american,italian


Number of artpieces with American culture made by non-american artist:  327


![outliers3.png](images/outliers3.png)


We are now looking at artworks created by non-American artists who, at some point in their lives, worked in the United States. Although the pieces are classified as American 'culture', we can see foreign motives and influence (elephants on dress, castles and knights on textile piec.



Now I will try to use both 'Country' and 'Artist Nationality' with 'Culture'.

In [23]:
df_cul_cou_art = df_cul_cou[~df_cul_cou['Artist Nationality'].str.contains('american', case=True, na=True, regex=False)].copy()

display(df_cul_cou_art.loc[[ 12037, 8160, 10968, 12083]])
print('Number of artpieces with American culture made outside of United States by an non-american artist: ', 
      df_cul_cou_art['Artist Nationality'].count())

,Object ID,Object Number,Object Name,Title,Culture,Country,Artist Nationality
12037,13111,54.90.108,Watercolor,The Bay of New York and Governors Island Taken...,american,ireland,irish
8160,8803,69.141.4,Tray,Tray,american,england,british
10968,11985,54.82,Watercolor,Osage Warrior,american,france,french
12083,13170,1970.9.15,Drawing,Wooded Landscape with Lake and Mountains (from...,american,germany,german


Number of artpieces with American culture made outside of United States by an non-american artist:  30


![outliers2.png](images/outliers2.png)

In the end, we have these examples of artworks that were made outside the United States, made by non-Americans (Irish, British, French and Geraan artists) and were inspired by American cult

From the original number of 29 463 pieces with American culture, we found 93 (0.32%) that were made outside the United States, 327 (1.11%) that were made by non-American artists, and 29 (0.098%) that were made outside the United States by non-Americanist.

These examples show nicely that outliers don't always have to be errors. They are simply rare cases that don't necessarily require corrections, but it's good we know about them and will consider them for her use.

Sadly, other cultures have very little artwork from outside their own countries. Most cases are related to datconsistency.

![image.jpg](images/outliers-others.jpg)

## Missing data

There is plenty of missing data. But those are only the ones that have a correct representation as missing. There still might be errors that are not shown (empty strings, placeholders, ...).

In [24]:
df.isnull().sum().sort_values(ascending=True)

Object Number                   0
Is Highlight                    0
Is Timeline Work                0
Is Public Domain                0
Object ID                       0
Repository                      0
Department                      0
Object End Date                 0
Link Resource                   0
Object Begin Date               0
Credit Line                   651
Object Name                  2266
AccessionYear                3862
Medium                       7215
Object Date                 13431
Title                       28803
Dimensions                  75058
Classification              78717
Constituent ID             202443
Artist Role                202443
Artist Prefix              202443
Artist Display Name        202443
Artist Alpha Sort          202443
Artist Nationality         202443
Artist Begin Date          202443
Artist End Date            202443
Artist Suffix              202491
Artist Display Bio         204533
Artist ULAN URL            257515
Artist Wikidat

### Artists

The features describing the artists who worked on the artwork have lots of incorrectly filled out data that should be missing. I will now try to properly assign missing d

**Artist WikidataURL**

Every value, depending on the number of artists, should contain at least one link to wikidata. Lots of them contain empty spaces with the seperator '|' for different artists. These kinds of rows should be considered as missing. A possible note, the number of separators indicates the number of artists working. By deleting these rows I could potentially lose that information. But thankfully, there are other features which also list all the artists, so I don't have to worry unless I start doing this to all the features abartists.

I am going to find all these values that don't have at least one link and change them to missing data.

In [25]:
column = 'Artist Wikidata URL'

total = len(df)
nul   = df[column].isna().sum()
n_nul =  df[column].notna().sum()

print('Missing data in:', column, '\n')
print(f"Null before:\t {nul} ({(nul * 100 / total):.0f}%)")
print(f"Non-null before: {n_nul} ({(n_nul * 100 / total):.0f}%)")

Missing data in: Artist Wikidata URL 

Null before:	 260754 (54%)
Non-null before: 224202 (46%)


In [26]:
df_tmp = df[column].copy

df_tmp = df.dropna(subset=[column])
df_tmp = df_tmp[~df_tmp[column].str.contains('wiki')]

drop_count = df_tmp[column].count()

print('Rows only with separators:' , drop_count, '\n')
display(df_tmp[column].value_counts().sort_values(ascending=False))

Rows only with separators: 10957 



Artist Wikidata URL
|                           8805
||                          1632
|||                          330
||||                         103
|||||                         44
||||||                        23
|||||||                        9
||||||||||                     3
|||||||||                      2
||||||||||||                   2
||||||||                       2
||||||||||||||||||||||||       1
||||||||||||||||||||||         1
Name: count, dtype: int64

In [27]:
tmp = df_tmp[column].unique()

df[column] = df[column].replace(tmp, np.nan)

print('Missing data in:', column, '\n')

print(f"Null before:\t {nul} ({(nul * 100 / total):.0f}%)")
print(f"Null after:\t {df[column].isna().sum()} ({(df[column].isna().sum() * 100 / total):.0f}%)\n")

print(f"Non-null before: {n_nul} ({(n_nul * 100 / total):.0f}%)")
print(f"Non-null after:  {df[column].notna().sum()} ({(df[column].notna().sum() * 100 / total):.0f}%)")

Missing data in: Artist Wikidata URL 

Null before:	 260754 (54%)
Null after:	 271711 (56%)

Non-null before: 224202 (46%)
Non-null after:  213245 (44%)


Now all values should contain at least one valid wikidata link and there should be no invalid missing data.

**Artist End Date**

Represent the year the artists died or stopped producing artworks. There are plenty of casses where years 2026 - 9999 seems to be used a placeholder while the artist or company is still active, I will replace them with a blank space. After that I will remove all the empty or other placesholders. I am going to use a similiar techinque I used with Artist Wikidata URL.

In [28]:
column = 'Artist End Date'

nul   = df[column].isna().sum()
n_nul =  df[column].notna().sum()

print('Missing data in:', column, '\n')
print(f"Null before:\t {nul} ({(nul * 100 / total):.0f}%)")
print(f"Non-null before: {n_nul} ({(n_nul * 100 / total):.0f}%)")

Missing data in: Artist End Date 

Null before:	 202443 (42%)
Non-null before: 282513 (58%)


In [29]:
df[['Object ID', 'Object Number', 'Object Name', 'Title', 'Artist Display Name', 'Artist Display Bio', 'Artist End Date']].loc[[89859, 125, 178222, 273061]]

,Object ID,Object Number,Object Name,Title,Artist Display Name,Artist Display Bio,Artist End Date
89859,156971,"2009.300.2174a, b",Bag (Chatelaine),Chatelaine,Gorham Manufacturing Company,"American, Providence, Rhode Island, 1831–present",9999
125,134,1972.187.1,Bedroom,Architectural elements from a Retiring Room in...,United Society of Believers in Christ’s Second...,"American, active ca. 1750–present",9999
178222,285578,2004.509.1,Photograph,"Street Speaker, 42nd Street, New York City",Benn Mitchell,"American, born 1926",9999
273061,406955,2013.547,Drawing Collection Ornament & Architecture,"Design for a Tazza, Bowl or Dish","Anonymous, German",,2050


![artist_end.png](images/artist_end.png)

The most common example of the 'Artist End Date' being higher than the current year is that the artist, or more commonly, the company which created these pieces is still active. The first two pictures are examples of this event as we can see in the 'Artist Display Bio' year of founding are still present.

But there are some cases which are errors. The photograph was made by Benn Mitchell who passed away in 2021, but the dataset hasn't been updated (last update was in November 2020). The drawing doesn't provide any information about the year, most likely it was just a placeholder.

In [30]:
df[column] = df[column].str.replace('202[6-9]|20[3-9][0-9]|2[1-9][0-9][0-9]|[3-9][0-9][0-9][0-9]| ', '', regex=True).str.strip()

df[column].value_counts().head(5)

Artist End Date
        44764
1975     8044
|        6544
1950     5341
2011     4545
Name: count, dtype: int64

In [31]:
df_tmp = df[column].copy

df_tmp = df.dropna(subset=[column])
df_tmp = df_tmp[~df_tmp[column].str.contains('\\d')]

drop_count = df_tmp[column].count()

print('Rows only with separators:' , drop_count, '\n')
display(df_tmp[column].value_counts().sort_values(ascending=False))

Rows only with separators: 52468 



Artist End Date
                            44764
|                            6544
||                            883
|||                           173
||||                           49
||||||                         33
|||||                          16
|||||||||                       2
|||||||                         2
||||||||||||||||||||||||        1
||||||||||||||||||||||          1
Name: count, dtype: int64

In [32]:
tmp = df_tmp[column].unique()

df[column] = df[column].replace(tmp, np.nan)

print('Missing data in:', column, '\n')

print(f"Null before:\t {nul} ({(nul * 100 / total):.0f}%)")
print(f"Null after:\t {df[column].isna().sum()} ({(df[column].isna().sum() * 100 / total):.0f}%)\n")

print(f"Non-null before: {n_nul} ({(n_nul * 100 / total):.0f}%)")
print(f"Non-null after:  {df[column].notna().sum()} ({(df[column].notna().sum() * 100 / total):.0f}%)")

Missing data in: Artist End Date 

Null before:	 202443 (42%)
Null after:	 254911 (53%)

Non-null before: 282513 (58%)
Non-null after:  230045 (47%)


Now all the values should contain only years up until the current year with no placeholders or empty cells.

**Medium - missing values**

The Medium feature needs to be prepared to be used in the KNN classification method. So in preparation for that, I have to deal with missing data first. There are some unnecessary symbols (\r\n, ?, (?), ...) Some values have incorrect representation (medium not available, no medium available, ...). I also made sure that the errors were valid, as the missing medium could simply imply that the original medium was lost (example: the text from a destroyed book was copied).

Some of the objects with 'no medium available' values.

![image.jpg](images/medium.jpg)

In [33]:
column = 'Medium'

nul   = df[column].isna().sum()
n_nul =  df[column].notna().sum()

print('Missing data in:', column, '\n')
print(f"Null before:\t {nul}\t ({(nul * 100 / total):.0f}%)")
print(f"Non-null before: {n_nul} ({(n_nul * 100 / total):.0f}%)")

Missing data in: Medium 

Null before:	 7215	 (1%)
Non-null before: 477741 (99%)


In [34]:
df['Medium'] = df['Medium'].str.strip()
df['Medium'] = df['Medium'].str.lower()

df['Medium'] = df['Medium'].str.replace('\r|\n|^ms$|\\\\', '', regex=True).str.strip()

df['Medium'] = df['Medium'].replace('no medium|unavailable|^unknown|medium not|not illustrated|^\\s*\\d+\\s*$|^[\\W_]+$', np.nan, regex=True)

In [35]:
print('Missing data in:', column, '\n')

print(f"Null before:\t {nul} ({(nul * 100 / total):.0f}%)")
print(f"Null after:\t {df[column].isna().sum()} ({(df[column].isna().sum() * 100 / total):.0f}%)\n")

print(f"Non-null before: {n_nul} ({(n_nul * 100 / total):.0f}%)")
print(f"Non-null after:  {df[column].notna().sum()} ({(df[column].notna().sum() * 100 / total):.0f}%)")

Missing data in: Medium 

Null before:	 7215 (1%)
Null after:	 8927 (2%)

Non-null before: 477741 (99%)
Non-null after:  476029 (98%)


That should cover all the missing data in 'Medium'.

**Medium - proper representation**

Before I try to fill in the missing data I would like better clean up the data in 'Medium' as I will use it with other features to fill in missing data.

The data in Medium is very inconsistent and still has some inproper representaion of values.

In [36]:
df['len']          = df['Medium'].str.len()
df['Medium_old']   = df['Medium'].copy()

In [37]:
top_n = 100
medium_rep = round(df['Medium'].value_counts().head(top_n).sum() * 100 / df['Medium'].count())
old_nuniq = df['Medium'].nunique()

title = df[df['len'] == df['len'].max()]['Title'].values[0]
print(f'Number of unique \'Medium\' values: {old_nuniq}')
print('The longest \'Medium\' entry:', title, ' with ', round(df['len'].max()), 'characters.\n')

display(df[['Object Name', 'Title', 'Medium', 'Classification']].loc[[204963,220784,283367,54724,7017,97615]])

print(f'The {top_n} most frequnet values in \'Medium\' represent {medium_rep}% of the dataset.')

#display(df['Medium'].value_counts().head(10))
#display(df['Medium'].value_counts().tail(10))

Number of unique 'Medium' values: 63754
The longest 'Medium' entry: Album of Drawings  with  7353 characters.



,Object Name,Title,Medium,Classification
204963,Drawing,Allegory of Art,"pen and black ink, brush and gray wash, height...",Drawings
220784,Book,Wilanowska Bajka,illustrations: offset color lithographs,Books
283367,Print,"Ste. Maria Salomé (St. Mary Salome), October 2...",etching; second state of two (lieure),Prints
54724,Teabowl,NaN,stoneware (hizen ware),Ceramics
7017,Side panel,Side panel from Brinsmaid House,"glass, wood",NaN
97615,Goblet,Scrolling design with griffin and a bear's mask,glass,Glass


The 100 most frequnet values in 'Medium' represent 56% of the dataset.


![MediumHeadTail.png](images/MediumHeadTail.png)

'Medium' has a very wide variety of values. To properly represent it, I need to reduce its descriptions. As I said in the beginning, I would like for 'Medium' to represent the main material it was created from or to help with the handling or storing of the art piece. 

Most entries follow a pattern of listing the materials or process of creation and the adding detail. I will try to remove these details that are often separated by symbols (;, :, ...), words (from, with, on, ...). Later I will remove all words that are only descriptive (blue, brown, carved, light, ...). 

![Medium.png](images/Medium.png)

The longest entry is an album with 250 pieces. In 'Medium', all the pieces are numbered by what was used in their creation.

In [38]:
df['Medium'] = df['Medium'].str.replace(';.*|\\..*|:.*|\\(.*\\)|\\?|from.*|.* on|\\w*-\\w*|with .*| in .*|.* of', '', regex=True).str.strip()
df['Medium'] = df['Medium'].str.replace('prints', 'print')
df['Medium'] = df['Medium'].str.replace('lithographs', 'lithograph')
df['Medium'] = df['Medium'].str.replace('water color', 'watercolor')

df['Medium'] = df['Medium'].str.replace('wood engraving|oak|mahogany|walnut|pine|maple|poplar', 'wood', regex=True)
df['Medium'] = df['Medium'].str.replace('silk / compound weave', 'silk')
df['Medium'] = df['Medium'].str.replace('engraving and etching', 'etching and engraving')

df['Medium'] = df['Medium'].str.replace('commercial color |commercial|, pigment', '', regex=True).str.strip() 
# just closer desriptions
df['Medium'] = df['Medium'].str.replace('\\b(?:a|adhered|applied|apparently|attached|aquamarine|base|beige|bobbin|black|blown|blue|brown|buff|carved|china|clear|cream|coarse|color|colorless|colored|coloured|covered|cut|dark|decorated|dye|dyed|embroidered|extremely|fine|fragment|glazed|gilded|gray|green|hand|heavy|hollow|india|indurated|ivory|knotted|laid|light|matte|medium|mixed|molded|modern|mottled|mosaic|natural|needle|orange|originally|pale|painted|partially|parian|partly|pink|plain|plated|possibly|pot|pressed|printed|probably|purple|red|salted|stained|stipple|stripped|tan|textured|trimmed|to|transfer|very|voided|weight|white|wove|wrapped|yellow)\\b', ' ', regex=True).str.strip()
# stripping after the first , might be a bit too much and will most likely will lower precision 
df['Medium'] = df['Medium'].str.replace(',.*', '', regex=True)

df['Medium'] = df['Medium'].str.replace('and and', 'and', regex=True).str.strip()
df['Medium'] = df['Medium'].str.replace('\\s+', ' ', regex=True).str.strip()

df['Medium'] = df['Medium'].replace('', np.nan, regex=True)
df['len']    = df['Medium'].str.len()

In this function I will try to absorb all the other values into the top 200 frequent values.

In [39]:
#print("Unique before reduction:", df['Medium'].nunique())
#df['Medium'].value_counts().head(30)

keep_k = 200
canon_freq = df['Medium'].value_counts(dropna=True)
top_k = list(canon_freq.index[:keep_k])

df['Medium_reduced'] = pd.NA

for label in top_k:
    mask = df['Medium'].str.contains(re.escape(label), case=False, na=False)
    df.loc[mask & df['Medium_reduced'].isna(), 'Medium_reduced'] = label

#print("Unique after reduction:", df['Medium_reduced'].nunique())
#print(df['Medium_reduced'].value_counts().head(30))

![mediumReduction.png](images/mediumReduction.png)

Now, I will only keep the top 99 values and the others will be changed to 'other'.

In [40]:
keep_k = 99
canon_freq = df['Medium'].value_counts(dropna=True)
top_k = list(canon_freq.index[:keep_k])

def reduce_cardinality(s):
    if pd.isna(s):
        return np.nan
    return s if s in top_k else 'other'

df['Medium_reduced'] = df['Medium'].apply(reduce_cardinality)

print('Comparison of original, cleaned and reduced versions of \'Medium\':\n')
df[['Medium_old', 'Medium','Medium_reduced']].sample(15)

Comparison of original, cleaned and reduced versions of 'Medium':



,Medium_old,Medium,Medium_reduced
431632,etching printed in red ink,etching,etching
53597,carnelian agate,carnelian agate,other
473663,"copper alloy, gold, enamel, jade, steel",copper alloy,copper alloy
406394,terracotta,terracotta,terracotta
91442,linen,linen,linen
103649,gilded brass,brass,brass
385416,cut paper silhouette,paper silhouette,paper silhouette
97748,glass,glass,glass
408303,terracotta,terracotta,terracotta
97929,silver on base metal,metal,metal


**Medium - adding missing values**

I will try to use other features (Object Name, Title, Classification, ...) to help me with identifing the 'Medium' of artworks. 

In [41]:
display(df[['Object Number','Object Name', 'Title', 'Medium', 'Classification']].loc[[198163,132751,218860,312179,104403]])

,Object Number,Object Name,Title,Medium,Classification
198163,36.70.5,Furniture leg,"Furniture leg, lion's paw",NaN,Ivory/Bone-Sculpture
132751,36.90.2554,Ribbon,Ribbon,NaN,Textiles-Trimmings
218860,65.595.12,Book,Book of complete information about pianos,NaN,Books
312179,1975.1.1152,Glass,Liqueur glass,NaN,Glass
104403,17.190.450,Medallion,Nicolas Diczel,NaN,Sculpture-Miniature


![Imputation.png](images/Imputation.png)

From the example I can see that we could gain some values for imputation. But it doesn't seem that much reliable.

I wiil impute 'Medium' using the most common value in progressively broader groups:

- Classification and Object Name (I will also check if there are enough examples in that pair, so we don’t impute from a group with 1 example)

- Classification

- Global

In [42]:
min_group_count = 10

group_counts = (df[df['Medium_reduced'].notna()].groupby(['Classification','Object Name']).size().rename('cnt'))
group_counts = group_counts.reset_index()

In [43]:
pair_mode = (df[df['Medium_reduced'].notna()]
             .groupby(['Classification','Object Name'])['Medium_reduced']
             .agg(lambda x: x.mode().iat[0] if not x.mode().empty else np.nan)
             .reset_index().merge(group_counts, on=['Classification','Object Name']))

pair_mode     = pair_mode[pair_mode['cnt'] >= min_group_count]
pair_mode_map = {(row['Classification'], row['Object Name']): row['Medium_reduced'] for _, row in pair_mode.iterrows()}

In [44]:
class_counts = (df[df['Medium_reduced'].notna()].groupby('Classification').size())
class_mode   = (df[df['Medium_reduced'].notna()].groupby('Classification')['Medium_reduced']
               .agg(lambda x: x.mode().iat[0] if not x.mode().empty else np.nan))

min_class_count = 50
class_mode = class_mode[class_counts >= min_class_count].to_dict()

global_mode = df['Medium_reduced'].mode().iat[0]

In [45]:
def impute_medium(row):
    if pd.notna(row['Medium_reduced']):
        return row['Medium_reduced']
    key = (row.get('Classification'), row.get('Object Name'))
    if key in pair_mode_map:
        return pair_mode_map[key]
    cls = row.get('Classification')
    if cls in class_mode:
        return class_mode[cls]
    return global_mode

df['Medium_imputed'] = df.apply(impute_medium, axis=1)

before_missing = df['Medium_reduced'].isna().sum()
after_missing = df['Medium_imputed'].isna().sum()
n_filled = (df['Medium_reduced'].isna() & df['Medium_imputed'].notna()).sum()
print(f"Missing before reduction: {before_missing}")
print(f"Missing after imputation: {after_missing}")
print(f"Number of rows filled:    {n_filled}")

Missing before reduction: 13789
Missing after imputation: 0
Number of rows filled:    13789


In [46]:
sample_imputed = df[(df['Medium_reduced'].isna())].sample(min(50, n_filled))
cols_to_show = ['Classification','Object Name','Medium_old','Medium_reduced','Medium_imputed']
sample_imputed[cols_to_show].sample(10)

#before_counts = df['Medium_reduced'].value_counts(dropna=True).head(30)
#after_counts  = df['Medium_imputed'].value_counts(dropna=True).head(30)
#print("TOP before-imputation:\n", before_counts)
#print("\nTOP after-imputation:\n", after_counts)

,Classification,Object Name,Medium_old,Medium_reduced,Medium_imputed
83521,NaN,Gloves,NaN,NaN,other
10332,NaN,"Painting, miniature",watercolor on ivory,NaN,other
132587,Textiles-Trimmings,Ribbon,NaN,NaN,silk
82553,NaN,Turban,NaN,NaN,other
477681,NaN,NaN,NaN,NaN,other
312818,Medals,Medal,honey-stone,NaN,bronze
222771,Books,Book,NaN,NaN,illustrations
342278,NaN,"Statue, Amasis, military",meta-greywacke,NaN,other
133105,Textiles-Woven,Piece,NaN,NaN,silk
85544,NaN,Handkerchief,NaN,NaN,other


![mediumImputation.png](images/mediumImputation.png)

The most added value by imputation was 'Other'. Imputation method doen't seem that good. The 'Classification' and 'Object Name' most likely needed also some form of cleaning and validation.

## Dimensions

Just like 'Medium', 'Dimensions' has a vast number of types and patterns for representation of the same things.

Before I begin, I have to fix data consistency and missing values.

In [47]:
df['Dimensions'] = df['Dimensions'].astype('string').str.lower().str.strip()
df['len']        = df['Dimensions'].str.len()

df['Dimensions'] = df['Dimensions'].str.replace('×', 'x')
df['Dimensions'] = df['Dimensions'].str.replace('`|\r|\n|see jib', '', regex=True).str.strip()
df['Dimensions'] = df['Dimensions'].replace('^oban$|n\\.a\\.|12vo|^\\s*[a-z]+(?:\\s+[a-z]+)*\\s*$|^no meas\\.|^2 sizes$|^overall:$|vertical ōban|no dimensions', np.nan, regex=True)
df['Dimensions'] = df['Dimensions'].replace('', np.nan)

df['Dimensions_old']   = df['Dimensions'].copy()
df['Dimensions_clean'] = pd.Series(pd.NA, index=df.index, dtype="string")

display(df[['Object Number', 'Object Name' ,'Dimensions', 'len']].loc[[460099, 301564, 18621]])
len_df = df['Dimensions_old'].notna().sum() 
over_hund = len(df[df['len'] >= 100])
print(f'Number of rows with 100+ characters: {over_hund} ( {round(over_hund * 100 / len_df)} %)')

,Object Number,Object Name,Dimensions,len
460099,2018.856.20a–jjj,Tools,"hammer (a): l. 11 1/2 in. (29.21 cm); head, 3 ...",2232
301564,15.72.3,Bowl,15.72.3a: h. 3 5/16 in. (8.4 cm) w. 10 3/4 i...,647
18621,29.151.2a–s,Armor,wt. approx. 56 lb. (25.4 kg); helmet (a): h. 1...,2347


Number of rows with 100+ characters: 20254 ( 5 %)


![dimensions.png](images/dimensions.png)


Not all of the items in the gallery are individual pieces, some are collections, made up from many pieces or contain detailed and varying descriptions that make it really hard to gain information without having to create many individual cases. These cases are rare and by possibly excluding values with 100+ characters we would lose around 5% of the da.



As the first step, I will extract by mapping some values that are repeated a bit more often, but their pattern isn't repeated that much.

In [48]:
dimmensions_map = {
    '2 to 8 cm': '2 - 8 cm',
    'each: 7 × 5 in. (17.8 × 12.7 cm), or smaller': '17.8 × 12.7 cm',
    'each swatch either 5 x 3 in. or 6 x 5 in.': '13.97 x 10.16 cm',
    '35mm': '3.5 cm',
    '10.5 x 7 x 0.5 cm (4 1/8 x 2 3/4 x 3/16 in.) each': '10.5 x 7 x 0.5 cm',
    '7 ½ x 9 9/16 in. (page)': '19.05 x 24.29 cm',
    'approx. 9 x 14 cm (3 9/16 x 5 1/2 in. ) each': '9 x 14 cm',
    'l. 2 to 8 cm': '2 - 8 cm',
    '35mm each': '3.5 cm',
    '' : '',
}

id_map = df['Dimensions'].isin(dimmensions_map.keys())

df.loc[id_map, 'Dimensions_clean'] = df.loc[id_map, 'Dimensions'].map(dimmensions_map)
df.loc[id_map, 'Dimensions']       = np.nan

I am going to use the length of 'Dimensions' and to try and isolate patterns. This way, I will have a better way to control from which kinds of values I am trying to extract. Then I will use a different combination of lengths, search and extraction patterns, till I have cleaned the desired amount of dimensions. 

In [49]:
inch_fractions = {
    '¼': 0.25, '½': 0.5, '¾': 0.75,
    '⅐': 1/7, '⅑': 1/9, '⅒': 0.1,
    '⅓': 1/3, '⅔': 2/3,
    '⅕': 0.2, '⅖': 0.4, '⅗': 0.6, '⅘': 0.8,
    '⅙': 1/6, '⅚': 5/6,
    '⅛': 0.125, '⅜': 0.375, '⅝': 0.625, '⅞': 0.875,
}

def frac_to_float(frac_str):
    try:
        a, b = frac_str.split('/')
        return float(a) / float(b)
    except Exception:
        return None

def inch_to_cm(inch_values):
    vals_cm = [x * 2.54 for x in inch_values]
    fmt = f"{{:.2f}}"
    return " x ".join(fmt.format(v) for v in vals_cm) + " cm"


def mixed_to_float(s):
    s = str(s).strip()
    
    m = re.match(r'^\s*(\d+)\s*([{}])\s*$'.format(''.join(re.escape(k) for k in inch_fractions.keys())), s)
    if m:
        whole = float(m.group(1))
        frac = inch_fractions.get(m.group(2), 0.0)
        return whole + frac

    for uf, val in inch_fractions.items():
        if uf in s:
            s2 = s.replace(uf, ' ')
            try:
                whole = float(s2.strip())
                return whole + val
            except:
                return val

    m = re.match(r'^\s*(\d+)\s+(\d+/\d+)\s*$', s)
    if m:
        whole = float(m.group(1))
        frac = frac_to_float(m.group(2))
        return whole + (frac if frac is not None else 0.0)

    m = re.match(r'^\s*(\d+/\d+)\s*$', s)
    if m:
        frac = frac_to_float(m.group(1))
        return float(frac) if frac is not None else None

    m = re.search(r'([0-9]+(?:\.[0-9]+)?)', s)
    if m:
        return float(m.group(1))
    return None

In [50]:
# cm
mask = ((df['len'] >= 0) & (df['len'] < 200) & df['Dimensions'].str.contains(
    '\\(\\s*\\d+(?:[.,]\\d+)?\\s*x\\s*\\d+(?:[.,]\\d+)?\\s*cm', na=False, regex=True))
extracted = df.loc[mask, 'Dimensions'].str.extract(r'\(([^)]+cm[^)]*)\)', expand=False).str.strip()
df.loc[extracted.index, 'Dimensions_clean'] = extracted
df.loc[extracted.index, 'Dimensions'] = np.nan

# cm with ending parentheses
mask = ((df['len'] >= 0) & (df['len'] < 500) & df['Dimensions'].str.contains('\\(\\s*[^)]*cm\\s*\\)', na=False, regex=True))
extracted = df.loc[mask, 'Dimensions'].str.extract(r'\(([^)]+cm)\)', expand=False).str.strip()
df.loc[extracted.index, 'Dimensions_clean'] = extracted
df.loc[extracted.index, 'Dimensions'] = np.nan

# inches
inch_pattern = re.compile(
    r'^\s*'                                  
    r'([0-9]+(?:\s+[0-9/]+)?(?:[.,][0-9]+)?)'
    r'(?:\s*x\s*([0-9]+(?:\s+[0-9/]+)?(?:[.,][0-9]+)?))?'
    r'(?:\s*x\s*([0-9]+(?:\s+[0-9/]+)?(?:[.,][0-9]+)?))?'
    r'(?:\s*(?:in\.?|inches))'
    r'.*$', re.I)
mask = df['Dimensions'].notna() & df['Dimensions'].str.match(inch_pattern)
for idx, val in df.loc[mask, 'Dimensions'].items():
    m = inch_pattern.match(val)
    if not m:
        continue
    groups = [g for g in m.groups() if g is not None]
    parsed = []
    for g in groups:
        f = mixed_to_float(g)
        if f is None:
            parsed = []
            break
        parsed.append(f)
    if parsed:
        df.at[idx, 'Dimensions_clean'] = inch_to_cm(parsed)
        df.at[idx, 'Dimensions'] = np.nan

# mm/mm
mask = ((df['len'] >= 0) & (df['len'] < 500) & df['Dimensions'].str.contains(r'mm\s*/\s*\d+mm$', na=False, regex=True))
mm_pairs = df.loc[mask, 'Dimensions'].str.extract(r'^\s*([0-9]+(?:[.,][0-9]+)?)\s*mm\s*/\s*([0-9]+(?:[.,][0-9]+)?)\s*mm\s*$', expand=True)
if not mm_pairs.empty:
    mm_pairs = mm_pairs.astype(float) / 10.0
    extracted = mm_pairs.apply(
        lambda row: f"{row.iloc[0]:.1f} x {row.iloc[1]:.1f} cm",
    )
    df.loc[extracted.index, 'Dimensions_clean'] = extracted
    df.loc[extracted.index, 'Dimensions'] = np.nan

# rest
mask = ((df['len'] >= 0) & (df['len'] < 80) & df['Dimensions'].str.contains(r'cm', na=False, regex=True))
extracted = df.loc[mask, 'Dimensions'].str.extract(r'([0-9]+(?:[.,][0-9]+)?(?:\s*x\s*[0-9]+(?:[.,][0-9]+)?)*(?:\s*cm))', expand=False)
df.loc[extracted.index, 'Dimensions_clean'] = extracted
df.loc[extracted.index, 'Dimensions'] = np.nan

In [51]:
df[['Dimensions_old','Dimensions', 'Dimensions_clean']].loc[[2, 46491, 352614, 155261, 263395]]

n_nul = df['Dimensions_clean'].notna().sum()
print("Dimensions possibly extracted:", n_nul, '(', round(n_nul * 100 / len_df), '%)')

Dimensions possibly extracted: 397721 ( 97 %)


I have possibly extracted around 97% of all the non-missing dimensions. There are some mistakes, but those could be corrected by continuing to fine tune the various extraction methods or use the map if they are more common.